In [ ]:
!pip install riotwatcher tenacity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 1.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import time
import json
import os
import random
from datetime import datetime, timezone
from riotwatcher import LolWatcher, ApiError
from tenacity import retry, wait_fixed, stop_after_attempt

# --------------------------
# CONFIGURACIÓN
# --------------------------
API_KEY = "RGAPI-31133159-ec6d-4880-8070-9c31dc836947" # Wed, May 6th, 2026 @ 6:04am (PT)
watcher = LolWatcher(API_KEY)

# Filtros de Partida
QUEUE_ID = 420  # SoloQ
MATCHES_PER_PLAYER = 15
MIN_GAME_DURATION = 1200 # 25 min

# Mapeo de Regiones para Match-V5
PLATFORM_TO_REGION = {
    "KR": "asia", "JP1": "asia",
    "EUW1": "europe", "EUN1": "europe", "TR1": "europe", "RU": "europe",
    "NA1": "americas", "BR1": "americas", "LA1": "americas", "LA2": "americas",
    "OC1": "sea", "PH2": "sea", "SG2": "sea", "TH2": "sea", "TW2": "sea", "VN2": "sea"
}

# Rango de tiempo: Parche actual (2 días atrás)
DT_START = datetime(2026, 4, 20, tzinfo=timezone.utc)
dt_start_unix = int(DT_START.timestamp())
DT_END = datetime(2026, 4, 26, tzinfo=timezone.utc)
dt_end_unix = int(DT_END.timestamp())

# Archivos de persistencia
SEED_FILE = "players_apex_seed.json"
OUTPUT_CSV = "dataset_soloq_high_elo.csv"

@retry(wait=wait_fixed(2), stop=stop_after_attempt(3))
def safe_call(fn, *args, **kwargs):
    return fn(*args, **kwargs)

print("✅ Entorno configurado.")

✅ Entorno configurado.


In [ ]:
def collect_all_apex_players(platforms):
    collected = []
    seen = set()
    tiers = ["CHALLENGER", "GRANDMASTER", "MASTER"] # Incluimos Master para volumen

    for plat in platforms:
        reg_route = PLATFORM_TO_REGION.get(plat.upper(), "americas")
        for tier in tiers:
            try:
                print(f"Buscando {tier} en {plat}...")
                if tier == "CHALLENGER": league = safe_call(watcher.league.challenger_by_queue, plat, "RANKED_SOLO_5x5")
                elif tier == "GRANDMASTER": league = safe_call(watcher.league.grandmaster_by_queue, plat, "RANKED_SOLO_5x5")
                else: league = safe_call(watcher.league.masters_by_queue, plat, "RANKED_SOLO_5x5")

                entries = league.get("entries", [])
                for entry in entries:
                    puuid = entry.get('puuid')
                    if puuid and puuid not in seen:
                        seen.add(puuid)
                        collected.append((reg_route, puuid))
                print(f"  Total actual: {len(collected)} PUUIDs")
            except Exception as e:
                print(f"  Error en {tier} {plat}: {e}")

    with open(SEED_FILE, "w") as f:
        json.dump(collected, f)
    return collected

# Ejecución: Añade o quita regiones aquí
PLATFORMS = ["kr", "euw1", "na1", "br1"]
if not os.path.exists(SEED_FILE):
    collected_puuids = collect_all_apex_players(PLATFORMS)
else:
    with open(SEED_FILE, "r") as f:
        collected_puuids = json.load(f)
    print(f"📂 Semillas cargadas desde disco: {len(collected_puuids)} jugadores.")

Buscando CHALLENGER en kr...
  Total actual: 300 PUUIDs
Buscando GRANDMASTER en kr...
  Total actual: 1000 PUUIDs
Buscando MASTER en kr...
  Total actual: 11000 PUUIDs
Buscando CHALLENGER en euw1...
  Total actual: 11311 PUUIDs
Buscando GRANDMASTER en euw1...
  Total actual: 12324 PUUIDs
Buscando MASTER en euw1...
  Total actual: 22324 PUUIDs
Buscando CHALLENGER en na1...
  Total actual: 22465 PUUIDs
Buscando GRANDMASTER en na1...
  Total actual: 23180 PUUIDs
Buscando MASTER en na1...
  Total actual: 32032 PUUIDs
Buscando CHALLENGER en br1...
  Total actual: 32195 PUUIDs
Buscando GRANDMASTER en br1...
  Total actual: 32831 PUUIDs
Buscando MASTER en br1...
  Total actual: 40590 PUUIDs


In [ ]:
def extract_compact_vector(m):
    info = m.get("info", {})
    if info.get("gameDuration", 0) < MIN_GAME_DURATION:
        return None

    participants = info.get("participants", [])

    # Extraer y ORDENAR IDs (Crucial para ML)
    blue_picks = sorted([p['championId'] for p in participants if p['teamId'] == 100])
    red_picks = sorted([p['championId'] for p in participants if p['teamId'] == 200])

    if len(blue_picks) != 5 or len(red_picks) != 5:
        return None

    # Blue wins = 1, Red wins = 0
    blue_win = 1 if any(p['win'] for p in participants if p['teamId'] == 100) else 0
    return blue_picks + red_picks + [blue_win]

print("⚙️ Lógica de transformación lista.")

⚙️ Lógica de transformación lista.


In [ ]:
final_vectors = []
processed_matches = set()

# Cargar progreso previo si existe el CSV
if os.path.exists(OUTPUT_CSV):
    df_old = pd.read_csv(OUTPUT_CSV)
    final_vectors = df_old.values.tolist()
    print(f"📈 Resumiendo desde {len(final_vectors)} partidas previas.")

print(f"🚀 Iniciando descarga para {len(collected_puuids)} fuentes...")

try:
    for i, (reg_route, puuid) in enumerate(collected_puuids):
        try:
            match_ids = safe_call(
                watcher.match.matchlist_by_puuid,
                reg_route, puuid,
                queue=QUEUE_ID,
                start_time=dt_start_unix,
                end_time=dt_end_unix,
                count=MATCHES_PER_PLAYER
            )

            if not match_ids: continue

            for mid in match_ids:
                if mid in processed_matches: continue
                processed_matches.add(mid)

                time.sleep(1.2) # Rate limit para Dev Key
                match_data = safe_call(watcher.match.by_id, reg_route, mid)

                vector = extract_compact_vector(match_data)
                if vector:
                    final_vectors.append(vector)

                # Guardar cada 50 partidas para no perder nada
                if len(final_vectors) % 50 == 0:
                    cols = [f'b{j}' for j in range(1,6)] + [f'r{j}' for j in range(1,6)] + ['win']
                    pd.DataFrame(final_vectors, columns=cols).to_csv(OUTPUT_CSV, index=False)

            if (i + 1) % 10 == 0:
                print(f"✅ Jugadores: {i+1}/{len(collected_puuids)} | Partidas Únicas: {len(final_vectors)}")

        except Exception as e:
            # Si hay error de Rate Limit 429, esperamos más tiempo
            if "429" in str(e):
                print("⏳ Límite alcanzado. Esperando 60s...")
                time.sleep(60)
            continue

except KeyboardInterrupt:
    print("\n🛑 Pausado por el usuario.")

# Guardado final al terminar todo
cols = [f'b{j}' for j in range(1,6)] + [f'r{j}' for j in range(1,6)] + ['win']
pd.DataFrame(final_vectors, columns=cols).to_csv(OUTPUT_CSV, index=False)
print(f"📦 Dataset finalizado: {len(final_vectors)} partidas.")

🚀 Iniciando descarga para 40590 fuentes...
✅ Jugadores: 10/40590 | Partidas Únicas: 63
✅ Jugadores: 20/40590 | Partidas Únicas: 137
✅ Jugadores: 30/40590 | Partidas Únicas: 217
✅ Jugadores: 40/40590 | Partidas Únicas: 282
✅ Jugadores: 50/40590 | Partidas Únicas: 338
✅ Jugadores: 60/40590 | Partidas Únicas: 404
✅ Jugadores: 70/40590 | Partidas Únicas: 460
✅ Jugadores: 80/40590 | Partidas Únicas: 514
✅ Jugadores: 90/40590 | Partidas Únicas: 547
✅ Jugadores: 100/40590 | Partidas Únicas: 606
✅ Jugadores: 110/40590 | Partidas Únicas: 647
✅ Jugadores: 120/40590 | Partidas Únicas: 683
✅ Jugadores: 130/40590 | Partidas Únicas: 738
✅ Jugadores: 140/40590 | Partidas Únicas: 773
✅ Jugadores: 150/40590 | Partidas Únicas: 813
✅ Jugadores: 160/40590 | Partidas Únicas: 852
✅ Jugadores: 170/40590 | Partidas Únicas: 872
✅ Jugadores: 180/40590 | Partidas Únicas: 907
✅ Jugadores: 190/40590 | Partidas Únicas: 942
✅ Jugadores: 200/40590 | Partidas Únicas: 979
✅ Jugadores: 210/40590 | Partidas Únicas: 1015
